In [6]:
# ==========================================================
# Import Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================================
# Machine Learning Tools
# ==========================================================

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ==========================================================
# Linear Models
# ==========================================================

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

# ==========================================================
# Tree-Based Models
# ==========================================================

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from lightgbm import LGBMRegressor

# ==========================================================
# Visualization
# ==========================================================

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12,6)

print("Libraries imported successfully!")

Libraries imported successfully!


In [14]:
# ==========================================================
# Summary
# In this step, I loaded the preprocessed training and test
# datasets. Then, I applied log transformation to the target
# variable because the competition uses RMSLE as the
# evaluation metric.
# ==========================================================

# ==========================================================
# Load Preprocessed Datasets
# ==========================================================

X = pd.read_csv("../Outputs/pipeline1_train_preprocessed.csv")

y_raw = pd.read_csv(
    "../Outputs/pipeline1_train_target.csv"
)["SalePrice"]

X_kaggle  = pd.read_csv(
    "../Outputs/pipeline1_test_preprocessed.csv"
)

print("Datasets loaded successfully!\n")

print(f"Training Features : {X.shape}")
print(f"Target Shape      : {y_raw.shape}")
print(f"Test Features     : {X_kaggle.shape}")

# ==========================================================
# Log Transform Target
# ==========================================================

y = np.log1p(y_raw)

print("\nTarget log transformation completed successfully!\n")

print("Original Target Statistics:")
display(y_raw.describe())

print("\nLog-Transformed Target Statistics:")
display(y.describe())

Datasets loaded successfully!

Training Features : (1460, 192)
Target Shape      : (1460,)
Test Features     : (1459, 192)

Target log transformation completed successfully!

Original Target Statistics:


count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64


Log-Transformed Target Statistics:


count    1460.000000
mean       12.024057
std         0.399449
min        10.460271
25%        11.775105
50%        12.001512
75%        12.273736
max        13.534474
Name: SalePrice, dtype: float64

In [13]:
# ==========================================================
# Train / Test Split
# ==========================================================

X_train, X_test_split, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train/Test Split completed!\n")

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test_split.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

Train/Test Split completed!

X_train : (1168, 192)
X_test  : (292, 192)
y_train : (1168,)
y_test  : (292,)


In [15]:
# ==========================================================
# Summary
# StandardScaler is applied only for linear models
# (Linear Regression, Ridge, Lasso, ElasticNet).
# Tree-based models such as Random Forest, XGBoost,
# and LightGBM do not require feature scaling.
# ==========================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_split),
    columns=X_test_split.columns,
    index=X_test_split.index
)

print("Feature Scaling completed successfully!\n")

print(f"Scaled Train Shape : {X_train_scaled.shape}")
print(f"Scaled Test Shape  : {X_test_scaled.shape}")

Feature Scaling completed successfully!

Scaled Train Shape : (1168, 192)
Scaled Test Shape  : (292, 192)


In [8]:
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold
)

# ==========================================================
# Summary
# In this step, I created a 5-Fold Cross-Validation
# framework to evaluate machine learning models.
# Cross-validation provides a more reliable estimate
# of model performance than a single train/test split.
# ==========================================================

# ==========================================================
# Create K-Fold Cross Validation
# ==========================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-Fold Cross-Validation created successfully!")

5-Fold Cross-Validation created successfully!


In [9]:
# ==========================================================
# Cross-Validation Evaluation Function
# ==========================================================

def rmse_cv(model, X, y):

    scores = cross_val_score(
        model,
        X,
        y,
        scoring="neg_root_mean_squared_error",
        cv=kf
    )

    return -scores

In [10]:
# ==========================================================
# Store Model Results
# ==========================================================

results = {}

print("Results dictionary created successfully!")

print("\nReady to evaluate models using Cross-Validation.")

Results dictionary created successfully!

Ready to evaluate models using Cross-Validation.


In [ ]:
print(f"Number of folds : {kf.get_n_splits()}")

Number of folds : 5


In [17]:
print(type(X_train))
print(type(y_train))

print(X_train.shape)
print(y_train.shape)

<class 'pandas.DataFrame'>
<class 'pandas.Series'>
(1168, 192)
(1168,)


In [19]:
# ==========================================================
# Prepare Data for XGBoost
# ==========================================================

X_train_xgb = X_train.fillna(0)
X_test_xgb = X_test_split.fillna(0)
X_kaggle_xgb = X_kaggle.fillna(0)

print("Missing values filled successfully!")

print(f"Train Missing : {X_train_xgb.isnull().sum().sum()}")
print(f"Test Missing  : {X_test_xgb.isnull().sum().sum()}")
print(f"Kaggle Missing: {X_kaggle_xgb.isnull().sum().sum()}")

Missing values filled successfully!
Train Missing : 0
Test Missing  : 0
Kaggle Missing: 0


In [21]:
# ==========================================================
# Summary
# In this step, I trained an XGBoost regression model and
# evaluated its performance using 5-Fold Cross-Validation.
# After evaluation, I trained the final model on the entire
# training dataset and generated predictions for the Kaggle
# test dataset.
# ==========================================================

# ==========================================================
# Train XGBoost Model
# ==========================================================

xgb = XGBRegressor(

    n_estimators=3000,

    learning_rate=0.05,

    max_depth=4,

    subsample=0.7,

    colsample_bytree=0.7,

    reg_alpha=0.1,

    reg_lambda=1.0,

    random_state=42,

    n_jobs=-1,

    verbosity=0
)

# ==========================================================
# 5-Fold Cross Validation
# ==========================================================

scores = rmse_cv(
    xgb,
    X_train_xgb,
    y_train
)

results["XGBoost"] = scores

print("=" * 50)
print("Cross Validation Results")
print("=" * 50)

print(f"Mean RMSE : {scores.mean():.5f}")
print(f"Std RMSE  : {scores.std():.5f}")

# ==========================================================
# Train Final Model
# ==========================================================

xgb.fit(
    X_train_xgb,
    y_train
)

print("\nFinal XGBoost model trained successfully!")

# ==========================================================
# Validation Prediction
# ==========================================================

val_pred_log = xgb.predict(X_test_xgb)

val_pred = np.expm1(val_pred_log)

print("\nValidation prediction completed!")

print(f"Prediction Shape : {val_pred.shape}")

# ==========================================================
# Kaggle Prediction
# ==========================================================

kaggle_pred_log = xgb.predict(X_kaggle_xgb)

kaggle_pred = np.expm1(kaggle_pred_log)

print("\nKaggle prediction completed!")

print(f"Kaggle Prediction Shape : {kaggle_pred.shape}")

Cross Validation Results
Mean RMSE : 0.12651
Std RMSE  : 0.01572

Final XGBoost model trained successfully!

Validation prediction completed!
Prediction Shape : (292,)

Kaggle prediction completed!
Kaggle Prediction Shape : (1459,)
